# 02 — Feature preparation

For each row of BigSolDB 2.0:

1. Drop rows with missing `LogS(mol/L)`.
2. Compute RDKit descriptors for solute SMILES.
3. Compute RDKit descriptors for solvent SMILES.
4. Drop columns that are NaN or constant on the whole dataset.

Caches everything to `results/features.npz`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors

DATA = Path.cwd().parent / 'data' / 'BigSolDBv2.0.csv'
RES  = Path.cwd().parent / 'results'
RES.mkdir(parents=True, exist_ok=True)

TARGET = 'LogS(mol/L)'
T_COL  = 'Temperature_K'

## Load and drop missing target

In [ ]:
df = pd.read_csv(DATA)
n_raw = len(df)
df = df.dropna(subset=[TARGET]).reset_index(drop=True)
print(f'dropped {n_raw - len(df):,} rows with missing {TARGET}')
print(f'remaining: {len(df):,}')

## Compute RDKit descriptors

Cached per unique SMILES so we don't recompute for repeated solutes/solvents.

In [ ]:
descriptor_specs = list(Descriptors._descList)
descriptor_names = [name for name, _ in descriptor_specs]
print(f'n RDKit descriptors registered: {len(descriptor_specs)}')

In [ ]:
def compute_desc_map(smiles_list):
    unique = list(dict.fromkeys(smiles_list))
    out = {}
    for i, smi in enumerate(unique):
        mol = Chem.MolFromSmiles(smi) if isinstance(smi, str) else None
        if mol is None:
            out[smi] = np.full(len(descriptor_specs), np.nan)
            continue
        vec = np.empty(len(descriptor_specs), dtype=np.float64)
        for j, (_, fn) in enumerate(descriptor_specs):
            try:
                vec[j] = fn(mol)
            except Exception:
                vec[j] = np.nan
        out[smi] = vec
        if (i + 1) % 200 == 0:
            print(f'  processed {i + 1}/{len(unique)} unique SMILES')
    return out

In [ ]:
print(f'computing solute descriptors for {df["SMILES_Solute"].nunique():,} unique solutes')
solute_map = compute_desc_map(df['SMILES_Solute'].tolist())

print(f'computing solvent descriptors for {df["SMILES_Solvent"].nunique():,} unique solvents')
solvent_map = compute_desc_map(df['SMILES_Solvent'].tolist())

## Stack per-row, replace inf, prune columns

In [ ]:
X_sol  = np.vstack([solute_map[smi]  for smi in df['SMILES_Solute']])
X_solv = np.vstack([solvent_map[smi] for smi in df['SMILES_Solvent']])

X_sol  = np.where(np.isfinite(X_sol),  X_sol,  np.nan)
X_solv = np.where(np.isfinite(X_solv), X_solv, np.nan)

print(f'X_sol  raw shape: {X_sol.shape}')
print(f'X_solv raw shape: {X_solv.shape}')

In [ ]:
def prune_columns(X, tag):
    n_start = X.shape[1]
    keep_nan = ~np.isnan(X).any(axis=0)
    X = X[:, keep_nan]
    surviving = [n for n, k in zip(descriptor_names, keep_nan) if k]
    n_after_nan = X.shape[1]
    stds = X.std(axis=0)
    keep_const = stds > 0
    X = X[:, keep_const]
    surviving = [n for n, k in zip(surviving, keep_const) if k]
    n_after_const = X.shape[1]
    print(f'{tag}: {n_start} -> {n_after_nan} after NaN prune -> {n_after_const} after constant prune')
    return X, surviving

X_sol,  solute_names  = prune_columns(X_sol,  'solute')
X_solv, solvent_names = prune_columns(X_solv, 'solvent')

## Save features

In [ ]:
T = df[T_COL].to_numpy(dtype=np.float64)
y = df[TARGET].to_numpy(dtype=np.float64)
pair_str = df['SMILES_Solute'].astype(str) + '||' + df['SMILES_Solvent'].astype(str)
pair_id, pair_uniques = pd.factorize(pair_str)

np.savez_compressed(
    RES / 'features.npz',
    X_sol=X_sol.astype(np.float32),
    X_solv=X_solv.astype(np.float32),
    T=T.astype(np.float32),
    y=y.astype(np.float32),
    pair_id=pair_id.astype(np.int64),
    solute_names=np.array(solute_names),
    solvent_names=np.array(solvent_names),
)
print('saved features.npz')
print(f'  X_sol  {X_sol.shape}')
print(f'  X_solv {X_solv.shape}')
print(f'  T      {T.shape}')
print(f'  y      {y.shape}')
print(f'  unique pairs: {len(pair_uniques):,}')